In [4]:
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from environment import PingPongEnv
from agent import ActorCritic
from agent import collect_self_play_rollout
from agent import compute_gae
from agent import ppo_update

In [5]:
def train():
    os.makedirs("output/LeftPaddleTrain",exist_ok=True)
    os.makedirs("output/RightPaddleTrain",exist_ok=True)

    env = PingPongEnv(render=False)

    left_paddle_agent = ActorCritic()
    right_paddle_agent = ActorCritic()

    left_paddle_agent_optimizer = optim.Adam(left_paddle_agent.parameters(), lr=3e-4)
    right_paddle_agent_optimizer = optim.Adam(right_paddle_agent.parameters(),lr=3e-4)

    total_iters = 500
    step_per_rollout = 2048

    print("Starting Training (this one will be it).......")
    print('-' * 60)

    total_timesteps = 0

    for iteration in range(total_iters):
        left_paddle_agent_transitions, right_paddle_agent_transitions = collect_self_play_rollout(left_paddle_agent,right_paddle_agent,env,step_per_rollout)

        total_timesteps += len(left_paddle_agent_transitions)

        left_paddle_agent_ep_rewards = []
        left_paddle_agent_ep_lengths = []
        left_paddle_agent_ep_reward = 0
        left_paddle_agent_ep_length = 0

        right_paddle_agent_ep_rewards = []
        right_paddle_agent_ep_lengths = []
        right_paddle_agent_ep_reward = 0
        right_paddle_agent_ep_length = 0




        for left_paddle_t, right_paddle_t in zip(left_paddle_agent_transitions, right_paddle_agent_transitions):
            left_paddle_agent_ep_reward += left_paddle_t["reward"]
            right_paddle_agent_ep_reward += right_paddle_t["reward"]

            left_paddle_agent_ep_length += 1
            right_paddle_agent_ep_length += 1


            if left_paddle_t["terminated"]:
                left_paddle_agent_ep_rewards.append(left_paddle_agent_ep_reward)
                left_paddle_agent_ep_lengths.append(left_paddle_agent_ep_length)

                left_paddle_agent_ep_reward = 0
                left_paddle_agent_ep_length = 0

            if right_paddle_t["terminated"]:
                right_paddle_agent_ep_rewards.append(right_paddle_agent_ep_reward)
                right_paddle_agent_ep_lengths.append(right_paddle_agent_ep_length)

                right_paddle_agent_ep_reward = 0
                right_paddle_agent_ep_length = 0

        left_paddle_advantages, left_paddle_returns = compute_gae(left_paddle_agent_transitions) #how do i do that last value in there
        right_paddle_advantages, right_paddle_returns = compute_gae(right_paddle_agent_transitions) #how do i do that last value in there

        left_paddle_metrics = ppo_update(
            left_paddle_agent,left_paddle_agent_optimizer,left_paddle_agent_transitions,left_paddle_advantages,left_paddle_returns
        )
        right_paddle_metrics = ppo_update(
            right_paddle_agent,right_paddle_agent_optimizer,right_paddle_agent_transitions,right_paddle_advantages,right_paddle_returns
        )



        with torch.no_grad():
            left_paddle_agent_obs = torch.FloatTensor(
                np.array([t["obs"] for t in left_paddle_agent_transitions])
            )
            right_paddle_agent_obs = torch.FloatTensor(
                np.array([t["obs"] for t in right_paddle_agent_transitions])
            )

            _, updated_values_left_agent = left_paddle_agent(left_paddle_agent_obs)
            _, updated_values_right_agent = right_paddle_agent(right_paddle_agent_obs)


        return_values_left_agent = left_paddle_returns.cpu().numpy()
        return_values_right_agent = right_paddle_returns.cpu().numpy()

        updated_values_np_left_agent = updated_values_left_agent.cpu().numpy()
        updated_values_np_right_agent = updated_values_right_agent.cpu().numpy()

        var_returns_left_agent = np.var(return_values_left_agent)
        var_returns_right_agent = np.var(return_values_right_agent)


        if var_returns_left_agent < 1e-6:
            explained_variance_left_agent = 0.0

        else:
            explained_variance_left_agent = 1 - np.var(return_values_left_agent - updated_values_np_left_agent)/var_returns_left_agent

        if var_returns_right_agent < 1e-6:
            explained_variance_right_agent = 0.0

        else:
            explained_variance_right_agent = 1 - np.var(return_values_right_agent - updated_values_np_right_agent)/var_returns_right_agent

        mean_reward_left_agent = np.mean(left_paddle_agent_ep_rewards) if left_paddle_agent_ep_rewards else 0
        mean_ep_len_left_agent = np.mean(left_paddle_agent_ep_lengths) if left_paddle_agent_ep_lengths else 0

        mean_reward_right_agent = np.mean(right_paddle_agent_ep_rewards) if right_paddle_agent_ep_rewards else 0
        mean_ep_len_right_agent = np.mean(right_paddle_agent_ep_lengths) if right_paddle_agent_ep_lengths else 0

        
        frac = 1.0 - iteration / total_iters
        lr = 3e-4 * frac

        for param_group in left_paddle_agent_optimizer.param_groups:
            param_group["lr"] = lr           


        for param_group in right_paddle_agent_optimizer.param_groups:
            param_group["lr"] = lr 

        print('-'*30)
        print("Left Paddle")
        print('-'*30)
        print(
        
            f"iterate: {iteration + 1:2d}/{total_iters} | "
            f"Number of rounds: {len(left_paddle_agent_ep_rewards):3d} | "
            f"Average reward: {mean_reward_left_agent:6.1f} | "
            f"KL: {left_paddle_metrics['approx_kl']:.4f} | "
            f"clip%: {left_paddle_metrics['clip_fraction']:.1%}"
        )

        print('-'*30)
        print("Right Paddle")
        print('-'*30)
        print(
        
            f"iterate: {iteration + 1:2d}/{total_iters} | "
            f"Number of rounds: {len(right_paddle_agent_ep_rewards):3d} | "
            f"Average reward: {mean_reward_right_agent:6.1f} | "
            f"KL: {right_paddle_metrics['approx_kl']:.4f} | "
            f"clip%: {right_paddle_metrics['clip_fraction']:.1%}"
        )


        eval_rewards_left = []
        eval_rewards_right = []

    if (iteration + 1) % 5 == 0:
        for _ in range(5):
            env.reset()
            left_paddle_agent_obs = env.get_player_observation("left_paddle_agent")
            right_paddle_agent_obs = env.get_player_observation("right_paddle_agent")
            done= False
            left_score = 0
            right_score = 0

            while not done:
            # NEW    
                left_obs_tensor = torch.FloatTensor(left_paddle_agent_obs)  # NEW
                right_obs_tensor = torch.FloatTensor(right_paddle_agent_obs)    # NEW

                with torch.no_grad():
                    left_action,_,_ = left_paddle_agent.get_action(
                        left_obs_tensor,
                        determinisitc=False
                    )
                    right_action,_,_ = right_paddle_agent.get_action(
                        right_obs_tensor,
                        determinisitc=False
                    )

                    left_reward,right_reward,done = env.step(left_action.item(),right_action.item())


                    left_score += left_reward
                    right_score += right_reward

                    left_paddle_agent_obs = env.get_player_observation("left_paddle_agent")
                    right_paddle_agent_obs = env.get_player_observation("right_paddle_agent")

            eval_rewards_left.append(left_score)
            eval_rewards_right.append(right_score)

    std_reward_left = np.std(eval_rewards_left)
    std_reward_right = np.std(eval_rewards_right)
    mean_eval_left = np.mean(eval_rewards_left)
    mean_eval_right = np.mean(eval_rewards_right)

    print(f"Training Complete: Left Evaluation over 20 rounds : {mean_eval_left:.1f} +/- {std_reward_left}")

    print(f"Training Complete: Right Evaluation over 20 rounds : {mean_eval_right:.1f} +/- {std_reward_right}")


    torch.save(left_paddle_agent.state_dict(), "output/LeftPaddleTrain/test12.pth")
    torch.save(right_paddle_agent.state_dict(), "output/RightPaddleTrain/test12.pth")

    print(f"Models have been saved.")
    env.close()


In [6]:
if __name__ == "__main__":
    train()

Starting Training (this one will be it).......
------------------------------------------------------------
------------------------------
Left Paddle
------------------------------
iterate:  1/500 | Number of rounds:   3 | Average reward:    0.3 | KL: 0.0051 | clip%: 4.6%
------------------------------
Right Paddle
------------------------------
iterate:  1/500 | Number of rounds:   3 | Average reward:    1.0 | KL: 0.0127 | clip%: 7.2%
------------------------------
Left Paddle
------------------------------
iterate:  2/500 | Number of rounds:   3 | Average reward:    0.3 | KL: 0.0093 | clip%: 4.5%
------------------------------
Right Paddle
------------------------------
iterate:  2/500 | Number of rounds:   3 | Average reward:    1.3 | KL: 0.0116 | clip%: 4.3%
------------------------------
Left Paddle
------------------------------
iterate:  3/500 | Number of rounds:   3 | Average reward:    1.0 | KL: 0.0169 | clip%: 9.4%
------------------------------
Right Paddle
----------------